# Exploring Meta data

This notebook explores and describes the meta data.
- Finding a paper that is nice to highlight in a presentation.
- Proportions of categorical columns.
- Distribution of numerical columns.

### Settings

In [ ]:
experiment_name = "free_1000_251013_pest_PD"
corpus_only = True # Show meta data of only PMIDs that are in the corpus [True], or show meta data of all PMIDs of interest [False].

## Initialisation

In [ ]:
# meta
__author__ ="Jennefer Beenen"
__version__ = "1.0"
__email__ = "j.beenen@pl.hanze.nl"
__status__ = "Development"

### Imports

In [ ]:
# imports
from pathlib import Path
import pandas as pd
import numpy as np
from matplotlib_venn import venn2, venn2_circles
import matplotlib.pyplot as plt

### Load files

In [ ]:
filename = f'meta_{experiment_name}.csv'
data_folder = "../../data/"
current_year = 2025

In [ ]:
# load data
df = pd.read_csv(f'{data_folder}meta/{filename}', index_col=0)
df.info()

In [ ]:
# load data
filename = f'corpus_{experiment_name}'
df_corpus = pd.read_csv(f'../../data/corpus/{filename}.csv')
df_corpus.info()

### Add columns `is_pdf`, `is_xml`, `cited_count_per_year`

In [ ]:
# load pmids downloaded in `data/xml_papers`
local_pdfs = {int(f.stem) for f in Path(f'{data_folder}pdf_papers/').iterdir() if f.suffixes[0] == ".pdf"}
local_xmls = {int(f.stem) for f in Path(f'{data_folder}xml_papers/').iterdir() if f.suffixes[0] == ".xml"}

In [ ]:
# Add `is_pdf` and `is_xml` columns to meta data
df['is_pdf'] = df['pmid'].isin(local_pdfs)
df['is_xml'] = df['pmid'].isin(local_xmls)

#### Normalise `cited_by_count` (adding column `cited_count_per_year`)

Since papers published in the same year as `current_year` will give an `inf`. We solved this problem by first assigning the `citated_by_count` value to `cited_count_per_year`, and then calculate the citation count per year for papers that are older than `current_year`. 

In [ ]:
# Normalise `cited_by_count` (adding column `cited_count_per_year`)
df["cited_count_per_year"] = df['cited_by_count'].astype(float)
df.loc[df["pub_year"] != current_year, "cited_count_per_year"] = df['cited_by_count'] / (current_year - df['pub_year'])

## Proportions by the columns

In [ ]:
if corpus_only:
    df = df[df['pmid'].isin(df_corpus["paper_name"].unique())]

### General

In [ ]:
# Check if meta contains duplicate PMIDs
df[df.duplicated(subset="pmid")]

There are no duplicates

In [ ]:
# Get impression
df.tail()

### Categorical

For some reason, the run of experiment `free_251013_pest_PD` resulted in all papers where "`is_oa` == `False`" to be `False` for `is_accepted` and `is_published` as well. First I thought the problem was that `is_published` meant something else, thus I searched for a different way to find out if a paper was peer-reviewed.

After updating code of the `pmid2meta.py` pipe to also obtain the ISSN number of a paper, I noticed that this property of `is_accepted` and `is_published` had changed for "`is_oa` == `False`" papers... I also started to notice that previous generated meta files did not have the same problem...

Thus `is_published` still appears to be a good candidate to know if a paper is peer-reviewed..

In [ ]:
# Get overview of True/False ratio
overview = pd.concat(
    [
        df[col].value_counts(normalize=True).rename(col) \
        for col in ["is_oa", "is_accepted", "is_published", "is_retracted"]
    ],
    axis=1
)

overview.fillna(0, inplace=True)
overview = overview * 100
overview

In [ ]:
# Plot ratio
fig, ax = plt.subplots()
x_labels = ["open access", "is accepted", "is published", "is retracted"]

for idx in overview.index[::-1]:

    if idx:
        p = ax.bar(
            x_labels,
            overview.loc[idx].values.round(2),
            0.7, 
            label=idx,
            bottom=[0, 0, 0, 0],
            color= "blue",
            alpha= 0.5
        )

    else:
        p = ax.bar(
            x_labels,
            overview.loc[idx].values.round(2),
            0.7, 
            label=idx,
            bottom=overview.loc[True].values,
            color= "red",
            alpha= 0.5
        )
    ax.bar_label(p, label_type='center', fmt= "{:.2f}%\n")

# Remove non applicable axis
ax.set_yticks([])
plt.rcParams['axes.spines.left'] = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.bottom'] = False

# Tidy
plt.legend(bbox_to_anchor= (0.95, 1.2))
plt.title("PMID ratios\n")
plt.show()


#### Venn oa vs published

In [ ]:
# Get set of PMIDs that are True for "is_oa", "is_accepted", "is_published"
set_oa_true = set(df.loc[df['is_oa'] == True, "pmid"].to_numpy())
# set_accepted_true = set(df.loc[df['is_accepted'] == True, "pmid"].to_numpy())
set_published_true = set(df.loc[df['is_published'] == True, "pmid"].to_numpy())

In [ ]:
# Plot venn
venn2(
    subsets=[
        set_oa_true, 
        set_published_true
    ],
    set_labels=(
        "open access",
        "is published"
    ),
    set_colors=(
        "red",
        "blue"
    ),
    alpha=0.4
)

# Add contrast
venn2_circles(
    subsets=[
        set_oa_true, 
        set_published_true
    ],
    linewidth= 1,
    alpha= 0.2,
)

# Add title
plt.title("PMID sets")
plt.show()

## Distribution of numerical columns

In [ ]:
# Reset axis
plt.rcParams['axes.spines.left'] = True
plt.rcParams['axes.spines.right'] =True
plt.rcParams['axes.spines.top'] = True
plt.rcParams['axes.spines.bottom'] = True

# Selected columns
columns = ["pub_year", "cited_by_count", "referenced_count", "cited_count_per_year"]
titles = dict(zip(columns, ["Publication year", "Total citations", "Referenced", "Citations per year"]))
x_labels = dict(zip(columns, ["Year", "Citations", "Reference", "Citations"]))

# Plot
for col in columns:
        df[col].hist(
            
            bins= np.arange(df[col].min(), df[col].max()+1, 1)
        )

        plt.xlabel(x_labels[col])
        plt.ylabel('Count')
        plt.title(titles[col])
        plt.grid(False)
        plt.show()

~END~